
# Test Different Load Matrices and Bias Matrices for the Heuristic Lower Agent

This notebook tests how the **heuristic lower A3 controller** behaves for many combinations of:

- load matrix \(L_{i,s}\)
- global bias matrix \(B_{i,s}\)
- UE count matrix \(K_{i,s}\)
- mobility safety metrics: handover failure ratio and ping-pong ratio

Main rule:

\[
\text{positive offset} \Rightarrow \text{stay / retain}
\]

\[
\text{negative offset} \Rightarrow \text{go / offload}
\]

The notebook checks whether the heuristic:
1. gives negative offsets when the bias asks for offloading,
2. gives positive offsets when the bias asks for retaining,
3. avoids offloading to an already-loaded neighbor,
4. gradually stabilizes load without emptying the source or saturating neighbors.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

GNB_NAMES = ["gNB0", "gNB1", "gNB2"]
SLICE_TYPES = ["eMBB", "URLLC", "mMTC"]
VALID_A3_OFFSETS_DB = np.array([-6, -4, -2, 0, 2, 4, 6], dtype=float)

NEIGHBORS = {
    0: [1, 2],
    1: [0, 2],
    2: [0, 1],
}

L_TARGET = 0.60



## 1. Utility functions

These functions display matrices, quantize offsets, and convert arrays to dictionary formats expected by the heuristic.


In [ ]:

def normalize_slice_type(slice_type: str) -> str:
    raw = str(slice_type or "eMBB")
    for known in SLICE_TYPES:
        if raw.upper() == known.upper():
            return known
    return raw


def quantize_a3_offset(offset_db: float) -> float:
    value = float(np.clip(offset_db, VALID_A3_OFFSETS_DB[0], VALID_A3_OFFSETS_DB[-1]))
    distances = np.abs(VALID_A3_OFFSETS_DB - value)
    candidates = VALID_A3_OFFSETS_DB[np.isclose(distances, distances.min())]
    # Tie-break: choose smaller magnitude to avoid aggressive movement.
    return float(candidates[np.argmin(np.abs(candidates))])


def matrix_to_dict(matrix, cols=SLICE_TYPES):
    matrix = np.asarray(matrix, dtype=float)
    return {
        (i, cols[s]): float(matrix[i, s])
        for i in range(matrix.shape[0])
        for s in range(matrix.shape[1])
    }


def ue_matrix_to_dict(matrix, cols=SLICE_TYPES):
    matrix = np.asarray(matrix, dtype=float)
    return {
        (i, cols[s]): int(round(matrix[i, s]))
        for i in range(matrix.shape[0])
        for s in range(matrix.shape[1])
    }


def dataframe_matrix(matrix, rows=GNB_NAMES, cols=SLICE_TYPES):
    return pd.DataFrame(np.asarray(matrix, dtype=float), index=rows, columns=cols)


def plot_heatmap(matrix, title, rows=GNB_NAMES, cols=SLICE_TYPES, vmin=None, vmax=None):
    matrix = np.asarray(matrix, dtype=float)
    fig, ax = plt.subplots(figsize=(5.5, 3.8))
    im = ax.imshow(matrix, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks(range(len(cols)))
    ax.set_yticks(range(len(rows)))
    ax.set_xticklabels(cols)
    ax.set_yticklabels(rows)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center")
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


def print_matrix(name, matrix):
    print(f"\n{name}")
    display(dataframe_matrix(matrix))



## 2. Heuristic lower agent

This is a notebook-local version of the stabilizing heuristic.

It uses:

\[
raw\_offset = 6b_{i,s} + safety
\]

where the safety term increases the offset when the neighbor is crowded, overloaded, or mobility-unstable.

So even if \(b_{i,s}<0\), the offset may move from negative to zero or positive when the target neighbor is risky.


In [ ]:

class HeuristicLowerAgent:
    """
    Stabilizing rule-based lower A3 controller.

    Meaning:
        positive offset -> handover harder -> stay / retain
        negative offset -> handover easier -> go / offload
    """

    def __init__(
        self,
        gnb_id,
        neighbor_ids,
        slice_types=("eMBB", "URLLC", "mMTC"),
        alpha_k=2.0,
        kappa_target=0.6,
        alpha_load=8.0,
        load_target=0.60,
        load_soft_margin=0.05,
        alpha_hf=3.0,
        alpha_pp=1.5,
        eta=0.25,
        neutral_deadband=0.15,
        neutral_offset_db=0.0,
        min_aggressive_offset=-4.0,
        max_offset_db=6.0,
    ):
        self.gnb_id = int(gnb_id)
        self.neighbor_ids = tuple(int(n) for n in neighbor_ids)
        self.slice_types = tuple(normalize_slice_type(s) for s in slice_types)

        self.alpha_k = float(alpha_k)
        self.kappa_target = float(kappa_target)
        self.alpha_load = float(alpha_load)
        self.load_target = float(load_target)
        self.load_soft_margin = float(load_soft_margin)
        self.alpha_hf = float(alpha_hf)
        self.alpha_pp = float(alpha_pp)
        self.eta = float(np.clip(eta, 0.0, 1.0))
        self.neutral_deadband = max(0.0, float(neutral_deadband))
        self.neutral_offset_db = float(np.clip(neutral_offset_db, -6.0, 6.0))
        self.min_aggressive_offset = float(np.clip(min_aggressive_offset, -6.0, 0.0))
        self.max_offset_db = float(np.clip(max_offset_db, 0.0, 6.0))

        self.previous_offsets = {
            (self.gnb_id, neighbor_id, slice_type): 0.0
            for neighbor_id in self.neighbor_ids
            for slice_type in self.slice_types
        }

    def reset(self):
        for key in self.previous_offsets:
            self.previous_offsets[key] = 0.0

    def _read_bias(self, bias_row, slice_idx, slice_type):
        if isinstance(bias_row, dict):
            value = float(bias_row.get(slice_type, 0.0))
        else:
            row = np.asarray(bias_row, dtype=float).reshape(-1)
            value = float(row[slice_idx]) if slice_idx < row.size else 0.0
        return float(np.clip(value, -1.0, 1.0))

    def compute_offsets(
        self,
        bias_row,
        ue_counts,
        kmax,
        slice_loads=None,
        handover_failure_ratios=None,
        ping_pong_ratios=None,
    ):
        outputs = {}

        slice_loads = slice_loads or {}
        handover_failure_ratios = handover_failure_ratios or {}
        ping_pong_ratios = ping_pong_ratios or {}

        for slice_idx, slice_type in enumerate(self.slice_types):
            b_i_s = self._read_bias(bias_row, slice_idx, slice_type)
            source_load = float(slice_loads.get((self.gnb_id, slice_type), 0.0))

            for neighbor_id in self.neighbor_ids:
                key = (self.gnb_id, int(neighbor_id), slice_type)
                prev = float(self.previous_offsets.get(key, 0.0))

                neighbor_count = float(ue_counts.get((int(neighbor_id), slice_type), 0))
                denom = max(float(kmax.get(slice_type, 1.0)), 1e-9)
                kappa_j_s = neighbor_count / denom

                neighbor_load = float(slice_loads.get((int(neighbor_id), slice_type), 0.0))
                rhf = float(handover_failure_ratios.get(key, 0.0))
                rpp = float(ping_pong_ratios.get(key, 0.0))

                crowding = max(kappa_j_s - self.kappa_target, 0.0)
                ue_safety = self.alpha_k * crowding

                neighbor_over_target = max(
                    neighbor_load - (self.load_target - self.load_soft_margin),
                    0.0,
                )
                load_safety = self.alpha_load * neighbor_over_target

                mobility_safety = (
                    self.alpha_hf * max(rhf, 0.0)
                    + self.alpha_pp * max(rpp, 0.0)
                )

                safety_term = ue_safety + load_safety + mobility_safety

                if abs(b_i_s) <= self.neutral_deadband:
                    raw_offset = self.neutral_offset_db + safety_term

                elif b_i_s < 0:
                    base_offset = 6.0 * b_i_s
                    source_excess = max(source_load - self.load_target, 0.0)

                    # If source is already near target, reduce offload aggressiveness.
                    if source_excess <= 0.02:
                        base_offset = max(base_offset, -2.0)

                    raw_offset = base_offset + safety_term

                    # Do not offload to a neighbor that is already at/above target.
                    if neighbor_load >= self.load_target:
                        raw_offset = max(raw_offset, 0.0)

                    # Avoid -6 when neighbor is not clearly safe.
                    if neighbor_load > self.load_target - 0.15:
                        raw_offset = max(raw_offset, self.min_aggressive_offset)

                else:
                    raw_offset = 6.0 * b_i_s + safety_term

                proto_offset = (1.0 - self.eta) * raw_offset + self.eta * prev
                proto_offset = float(np.clip(proto_offset, -6.0, self.max_offset_db))
                applied_offset = float(quantize_a3_offset(proto_offset))

                self.previous_offsets[key] = applied_offset

                outputs[key] = {
                    "proto_offset_db": proto_offset,
                    "applied_offset_db": applied_offset,
                    "bias": float(b_i_s),
                    "source_load": float(source_load),
                    "neighbor_load": float(neighbor_load),
                    "safety_term_db": float(safety_term),
                    "ue_safety_db": float(ue_safety),
                    "load_safety_db": float(load_safety),
                    "mobility_safety_db": float(mobility_safety),
                    "handover_failure_ratio": float(rhf),
                    "ping_pong_ratio": float(rpp),
                    "neighbor_ue_fraction": float(kappa_j_s),
                }

        return outputs



## 3. Create one full heuristic system

There is one lower agent per serving gNB.


In [ ]:

def make_agents(**kwargs):
    return {
        gnb_id: HeuristicLowerAgent(
            gnb_id=gnb_id,
            neighbor_ids=NEIGHBORS[gnb_id],
            slice_types=SLICE_TYPES,
            **kwargs,
        )
        for gnb_id in range(len(GNB_NAMES))
    }


def reset_agents(agents):
    for agent in agents.values():
        agent.reset()


def compute_all_offsets(
    load_matrix,
    bias_matrix,
    ue_count_matrix=None,
    agents=None,
    kmax=None,
    hf=None,
    pp=None,
):
    load_matrix = np.asarray(load_matrix, dtype=float)
    bias_matrix = np.asarray(bias_matrix, dtype=float)

    if ue_count_matrix is None:
        ue_count_matrix = np.maximum(np.round(load_matrix * 30), 1).astype(int)

    if agents is None:
        agents = make_agents()

    if kmax is None:
        kmax = {"eMBB": 30.0, "URLLC": 20.0, "mMTC": 80.0}

    slice_loads = matrix_to_dict(load_matrix)
    ue_counts = ue_matrix_to_dict(ue_count_matrix)

    hf = hf or {}
    pp = pp or {}

    all_outputs = {}

    for i, agent in agents.items():
        bias_row = {
            slice_type: float(bias_matrix[i, s_idx])
            for s_idx, slice_type in enumerate(SLICE_TYPES)
        }
        outputs_i = agent.compute_offsets(
            bias_row=bias_row,
            ue_counts=ue_counts,
            kmax=kmax,
            slice_loads=slice_loads,
            handover_failure_ratios=hf,
            ping_pong_ratios=pp,
        )
        all_outputs.update(outputs_i)

    return all_outputs


def offsets_to_dataframe(outputs):
    rows = []
    for (src, dst, slice_type), info in outputs.items():
        rows.append({
            "source": f"gNB{src}",
            "target": f"gNB{dst}",
            "slice": slice_type,
            "bias": info["bias"],
            "source_load": info["source_load"],
            "neighbor_load": info["neighbor_load"],
            "proto_offset_db": info["proto_offset_db"],
            "applied_offset_db": info["applied_offset_db"],
            "safety_term_db": info["safety_term_db"],
            "load_safety_db": info["load_safety_db"],
            "ue_safety_db": info["ue_safety_db"],
            "mobility_safety_db": info["mobility_safety_db"],
        })
    return pd.DataFrame(rows).sort_values(["slice", "source", "target"]).reset_index(drop=True)


def plot_offsets_by_target(outputs):
    for target_id in range(len(GNB_NAMES)):
        mat = np.full((len(GNB_NAMES), len(SLICE_TYPES)), np.nan)
        for (src, dst, s), info in outputs.items():
            if dst == target_id:
                mat[src, SLICE_TYPES.index(s)] = info["applied_offset_db"]

        fig, ax = plt.subplots(figsize=(5.5, 3.8))
        im = ax.imshow(np.nan_to_num(mat, nan=0.0), vmin=-6, vmax=6)
        ax.set_title(f"Applied offsets toward gNB{target_id}")
        ax.set_xticks(range(len(SLICE_TYPES)))
        ax.set_yticks(range(len(GNB_NAMES)))
        ax.set_xticklabels(SLICE_TYPES)
        ax.set_yticklabels(GNB_NAMES)

        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                txt = "—" if np.isnan(mat[i, j]) else f"{mat[i, j]:.0f}"
                ax.text(j, i, txt, ha="center", va="center")

        plt.colorbar(im, ax=ax, label="A3 offset dB")
        plt.tight_layout()
        plt.show()



## 4. Define test cases

You can edit these matrices or add more cases.

Load matrix rows are gNBs, columns are slices:

\[
[eMBB, URLLC, mMTC]
\]

Bias matrix has the same shape:

- `-1`: offload
- `0`: neutral
- `+1`: retain


In [ ]:

TEST_CASES = {
    "case_1_gNB0_eMBB_offload_safe_neighbors": {
        "load": np.array([
            [0.88, 0.50, 0.50],
            [0.18, 0.50, 0.50],
            [0.48, 0.50, 0.50],
        ]),
        "bias": np.array([
            [-1.0, 0.0, 0.0],
            [ 1.0, 0.0, 0.0],
            [ 0.0, 0.0, 0.0],
        ]),
    },

    "case_2_gNB0_eMBB_offload_but_neighbors_loaded": {
        "load": np.array([
            [0.90, 0.50, 0.50],
            [0.72, 0.50, 0.50],
            [0.80, 0.50, 0.50],
        ]),
        "bias": np.array([
            [-1.0, 0.0, 0.0],
            [ 0.0, 0.0, 0.0],
            [ 0.0, 0.0, 0.0],
        ]),
    },

    "case_3_retain_gNB1_URLLC": {
        "load": np.array([
            [0.50, 0.50, 0.50],
            [0.50, 0.20, 0.50],
            [0.50, 0.90, 0.50],
        ]),
        "bias": np.array([
            [0.0, 0.0, 0.0],
            [0.0, 1.0, 0.0],
            [0.0, -1.0, 0.0],
        ]),
    },

    "case_4_multi_slice_conflict": {
        "load": np.array([
            [0.92, 0.86, 0.35],
            [0.32, 0.90, 0.88],
            [0.76, 0.30, 0.28],
        ]),
        "bias": np.array([
            [-1.0, -1.0,  0.0],
            [ 0.0, -1.0, -1.0],
            [-0.5,  0.5,  0.5],
        ]),
    },

    "case_5_all_neutral": {
        "load": np.array([
            [0.55, 0.50, 0.50],
            [0.50, 0.55, 0.50],
            [0.50, 0.50, 0.55],
        ]),
        "bias": np.zeros((3, 3)),
    },
}



## 5. Run one test case

Change `case_name` to test another scenario.


In [ ]:

case_name = "case_1_gNB0_eMBB_offload_safe_neighbors"

case = TEST_CASES[case_name]
load = case["load"]
bias = case["bias"]

agents = make_agents(load_target=L_TARGET)
outputs = compute_all_offsets(load, bias, agents=agents)
df = offsets_to_dataframe(outputs)

print("Selected case:", case_name)
print_matrix("Load matrix", load)
print_matrix("Bias matrix", bias)
display(df)

plot_heatmap(load, "Load matrix", vmin=0, vmax=1)
plot_heatmap(bias, "Bias matrix", vmin=-1, vmax=1)
plot_offsets_by_target(outputs)



## 6. Automatic verification table

This section checks the main expected behavior.


In [ ]:

def verify_outputs(outputs, load_target=L_TARGET):
    rows = []

    for (src, dst, slice_type), info in outputs.items():
        bias_val = info["bias"]
        offset = info["applied_offset_db"]
        source_load = info["source_load"]
        neighbor_load = info["neighbor_load"]

        if bias_val < -0.15 and source_load > load_target and neighbor_load < load_target:
            expected = "negative"
            passed = offset < 0
        elif bias_val < -0.15 and neighbor_load >= load_target:
            expected = "non-negative"
            passed = offset >= 0
        elif bias_val > 0.15:
            expected = "positive"
            passed = offset > 0
        elif abs(bias_val) <= 0.15 and neighbor_load < load_target:
            expected = "near neutral"
            passed = abs(offset) <= 2
        else:
            expected = "safe / non-aggressive"
            passed = offset >= -2

        rows.append({
            "source": f"gNB{src}",
            "target": f"gNB{dst}",
            "slice": slice_type,
            "bias": bias_val,
            "source_load": source_load,
            "neighbor_load": neighbor_load,
            "offset": offset,
            "expected": expected,
            "pass": bool(passed),
        })

    return pd.DataFrame(rows).sort_values(["slice", "source", "target"]).reset_index(drop=True)

verification_df = verify_outputs(outputs)
display(verification_df)

print("Passed checks:", verification_df["pass"].sum(), "/", len(verification_df))



## 7. Compare all predefined cases

This runs the heuristic on every load/bias case and summarizes the offsets.


In [ ]:

all_case_rows = []

for name, case in TEST_CASES.items():
    agents = make_agents(load_target=L_TARGET)
    outputs = compute_all_offsets(case["load"], case["bias"], agents=agents)
    df_case = verify_outputs(outputs)
    df_case.insert(0, "case", name)
    all_case_rows.append(df_case)

all_cases_df = pd.concat(all_case_rows, ignore_index=True)
display(all_cases_df)

summary = all_cases_df.groupby("case")["pass"].agg(["sum", "count"])
summary["pass_rate"] = summary["sum"] / summary["count"]
display(summary)



## 8. Simple load stabilization simulation

This is not the full radio simulator. It is a simplified sanity model.

At each step:

1. compute offsets from load and bias,
2. move small load amounts only from overloaded source to safe neighbors,
3. stop transfer if neighbor is near target,
4. track load variance.

This helps verify that the heuristic does not empty the source or saturate neighbors.


In [ ]:

def compute_rule_based_bias(load, target=L_TARGET, margin=0.10):
    load = np.asarray(load, dtype=float)
    bias = np.zeros_like(load)

    bias[load > target + margin] = -1.0
    bias[load < target - margin] = 1.0
    return bias


def simulate_load_transfer_step(
    load,
    bias,
    agents,
    max_transfer=0.04,
    target=L_TARGET,
):
    load = np.asarray(load, dtype=float).copy()
    outputs = compute_all_offsets(load, bias, agents=agents)

    delta = np.zeros_like(load)

    for (src, dst, slice_type), info in outputs.items():
        s_idx = SLICE_TYPES.index(slice_type)

        source_excess = max(load[src, s_idx] - target, 0.0)
        neighbor_capacity = max(target - load[dst, s_idx], 0.0)
        offset = info["applied_offset_db"]

        # Only negative offset can move load from source to target.
        if offset < 0 and source_excess > 0 and neighbor_capacity > 0:
            strength = abs(offset) / 6.0
            transfer = min(
                max_transfer * strength,
                source_excess,
                neighbor_capacity,
            )

            delta[src, s_idx] -= transfer
            delta[dst, s_idx] += transfer

    new_load = np.clip(load + delta, 0.0, 1.0)
    return new_load, outputs, delta


def load_variance(load):
    load = np.asarray(load, dtype=float)
    return float(np.mean(np.var(load, axis=0)))


def run_stabilization_simulation(
    initial_load,
    initial_bias=None,
    steps=30,
    use_dynamic_bias=True,
    target=L_TARGET,
):
    agents = make_agents(load_target=target)
    load = np.asarray(initial_load, dtype=float).copy()

    history = [load.copy()]
    bias_history = []
    variance_history = [load_variance(load)]
    outputs_history = []
    delta_history = []

    for t in range(steps):
        if use_dynamic_bias:
            bias = compute_rule_based_bias(load, target=target)
        else:
            bias = np.asarray(initial_bias, dtype=float).copy()

        new_load, outputs, delta = simulate_load_transfer_step(
            load=load,
            bias=bias,
            agents=agents,
            target=target,
        )

        bias_history.append(bias.copy())
        outputs_history.append(outputs)
        delta_history.append(delta.copy())
        load = new_load.copy()
        history.append(load.copy())
        variance_history.append(load_variance(load))

    return {
        "load_history": np.array(history),
        "bias_history": np.array(bias_history),
        "variance_history": np.array(variance_history),
        "outputs_history": outputs_history,
        "delta_history": np.array(delta_history),
    }



## 9. Run stabilization on a selected case


In [ ]:

case_name = "case_1_gNB0_eMBB_offload_safe_neighbors"
initial_load = TEST_CASES[case_name]["load"]
initial_bias = TEST_CASES[case_name]["bias"]

result = run_stabilization_simulation(
    initial_load=initial_load,
    initial_bias=initial_bias,
    steps=30,
    use_dynamic_bias=True,  # set False to test the fixed given bias matrix
)

load_history = result["load_history"]
variance_history = result["variance_history"]

print_matrix("Initial load", load_history[0])
print_matrix("Final load", load_history[-1])

plot_heatmap(load_history[0], "Initial load", vmin=0, vmax=1)
plot_heatmap(load_history[-1], "Final load after heuristic stabilization", vmin=0, vmax=1)

plt.figure(figsize=(7, 4))
for g in range(3):
    plt.plot(load_history[:, g, 0], marker="o", label=f"{GNB_NAMES[g]} eMBB")
plt.axhline(L_TARGET, linestyle="--", label="target")
plt.title("eMBB load evolution")
plt.xlabel("step")
plt.ylabel("load")
plt.ylim(0, 1)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(variance_history, marker="o")
plt.title("Mean slice-load variance")
plt.xlabel("step")
plt.ylabel("variance")
plt.grid(True)
plt.tight_layout()
plt.show()

print("Initial variance:", variance_history[0])
print("Final variance:", variance_history[-1])



## 10. Interactive custom matrices

Edit `custom_load` and `custom_bias`, then run the next cells.

Bias meaning:

- `-1.0`: strong offload
- `-0.5`: soft offload
- `0.0`: neutral
- `+0.5`: soft retain
- `+1.0`: strong retain


In [ ]:

custom_load = np.array([
    [0.90, 0.40, 0.50],
    [0.20, 0.85, 0.50],
    [0.40, 0.30, 0.90],
])

custom_bias = np.array([
    [-1.0,  0.0,  0.0],
    [ 0.0, -1.0,  0.0],
    [ 0.0,  0.5, -1.0],
])

agents = make_agents(load_target=L_TARGET)
custom_outputs = compute_all_offsets(custom_load, custom_bias, agents=agents)
custom_df = offsets_to_dataframe(custom_outputs)
custom_verification = verify_outputs(custom_outputs)

print_matrix("Custom load", custom_load)
print_matrix("Custom bias", custom_bias)
display(custom_df)
display(custom_verification)

plot_heatmap(custom_load, "Custom load", vmin=0, vmax=1)
plot_heatmap(custom_bias, "Custom bias", vmin=-1, vmax=1)
plot_offsets_by_target(custom_outputs)


In [ ]:

custom_result = run_stabilization_simulation(
    initial_load=custom_load,
    initial_bias=custom_bias,
    steps=30,
    use_dynamic_bias=False,  # fixed custom bias
)

custom_history = custom_result["load_history"]
custom_variance = custom_result["variance_history"]

print_matrix("Custom initial load", custom_history[0])
print_matrix("Custom final load", custom_history[-1])

plot_heatmap(custom_history[0], "Custom initial load", vmin=0, vmax=1)
plot_heatmap(custom_history[-1], "Custom final load", vmin=0, vmax=1)

plt.figure(figsize=(7, 4))
for s_idx, slice_type in enumerate(SLICE_TYPES):
    plt.plot(custom_history[:, :, s_idx].mean(axis=1), marker="o", label=f"{slice_type} mean load")
plt.axhline(L_TARGET, linestyle="--", label="target")
plt.title("Mean load per slice")
plt.xlabel("step")
plt.ylabel("load")
plt.ylim(0, 1)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(custom_variance, marker="o")
plt.title("Custom case: mean slice-load variance")
plt.xlabel("step")
plt.ylabel("variance")
plt.grid(True)
plt.tight_layout()
plt.show()

print("Initial variance:", custom_variance[0])
print("Final variance:", custom_variance[-1])



## 11. Random stress test

This creates random load and bias matrices and checks whether the heuristic produces safe signs.

You can increase `N_RANDOM_TESTS`.


In [ ]:

def random_bias_from_load(load, target=L_TARGET, margin=0.10, noise_prob=0.15, rng=None):
    rng = rng or np.random.default_rng(0)
    bias = compute_rule_based_bias(load, target=target, margin=margin)

    # Add some random imperfect bias decisions to stress-test the heuristic.
    mask = rng.random(size=bias.shape) < noise_prob
    random_values = rng.choice([-1.0, -0.5, 0.0, 0.5, 1.0], size=bias.shape)
    bias[mask] = random_values[mask]
    return bias


def run_random_stress_test(n_tests=50, seed=42):
    rng = np.random.default_rng(seed)
    rows = []

    for test_id in range(n_tests):
        load = rng.uniform(0.10, 0.95, size=(3, 3))
        bias = random_bias_from_load(load, rng=rng)

        agents = make_agents(load_target=L_TARGET)
        outputs = compute_all_offsets(load, bias, agents=agents)
        verify_df = verify_outputs(outputs)

        rows.append({
            "test_id": test_id,
            "passed": int(verify_df["pass"].sum()),
            "total": int(len(verify_df)),
            "pass_rate": float(verify_df["pass"].mean()),
            "initial_variance": load_variance(load),
        })

    return pd.DataFrame(rows)


N_RANDOM_TESTS = 100
stress_df = run_random_stress_test(N_RANDOM_TESTS)
display(stress_df.head())
display(stress_df.describe())

plt.figure(figsize=(7, 4))
plt.hist(stress_df["pass_rate"], bins=10)
plt.title("Random stress test pass-rate distribution")
plt.xlabel("pass rate")
plt.ylabel("count")
plt.grid(True)
plt.tight_layout()
plt.show()



# Conclusion

This notebook lets you test the heuristic with:

1. predefined load/bias scenarios,
2. your own custom matrices,
3. simplified stabilization over time,
4. random stress tests.

The expected behavior is:

- overloaded source + negative bias + safe neighbor → negative offset,
- loaded or risky neighbor → offset moves toward zero or positive,
- positive bias → positive offset,
- neutral bias → near-zero offset,
- load variance should decrease in the simplified stabilization simulation.
